In [6]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
import numpy as np
from src.utils import *
from src.params import *
import json
from src.preprocess import *
from src.dataset import *

import torch
from torch import nn
from torch.utils.data import Dataset, Subset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from pytorch_lightning import LightningDataModule, LightningModule
from torchmetrics import KLDivergence


# Design baseline model
- simple CNN with VGG-ish architecture
- hardcoded because objective isnt to fine tune that

## Conv block and model

In [2]:
class ConvBlock(nn.Module):

    def __init__(self, in_channels, out_channels, stride= 1, kernel= 3, padding= 1):

        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels= in_channels,
                      out_channels= out_channels,
                      stride= stride,
                      padding= padding,
                      kernel_size= kernel,
                      bias= False),
            nn.BatchNorm2d(num_features= out_channels),
            nn.ReLU()
        )

    def forward(self,x):
        return self.block(x)

In [3]:
class BaselineModel(nn.Module):

    def __init__(self, n_classes, n_channels):

        super().__init__()
        self.n_classes = n_classes
        self.n_channels = n_channels

        self.block1 = nn.Sequential(
            ConvBlock(in_channels= self.n_channels,
                      out_channels= 32),
            ConvBlock(in_channels= 32,
                      out_channels= 64),
            nn.MaxPool2d(kernel_size= 3, stride= 2, padding= 1),
            nn.Dropout(0.3)
        )

        self.block2 = nn.Sequential(
            ConvBlock(in_channels= 64,
                      out_channels= 128),
            ConvBlock(in_channels= 128,
                      out_channels= 256),
            nn.MaxPool2d(kernel_size= 3, stride= 2, padding= 1),
            nn.Dropout(0.3)
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Linear(256, 64),
            nn.Dropout(0.3),
            nn.Linear(64, self.n_classes)
        )

    def forward(self,x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.head(x)

        return nn.functional.log_softmax(x, dim= 1)


In [ ]:
#test forward pass
test = BaselineModel(n_channels= 4, n_classes= 6)
x = torch.randn(1,4, 100, 300)
logits = test(x)
print(logits)
print(logits.exp().sum())

tensor([[-2.0516, -1.6794, -1.3408, -1.3039, -2.5184, -2.6408]],
       grad_fn=<LogSoftmaxBackward0>)
tensor(1.0000, grad_fn=<SumBackward0>)


# Lighntning

## lightning module

In [ ]:
class BrainLightning(LightningModule):

    def __init__(self, model, n_classes= 6, lr=1e-3):

        super().__init__()
        self.save_hyperparameters(ignore= ["model"])
        self.model = model
        #the model returns y_pred as log_proba, but y_true is proba. This is expected for the loss
        self.criterion = nn.KLDivLoss(reduction= "batchmean")
        self.train_kl = KLDivergence(reduction= "mean")
        self.val_kl = KLDivergence(reduction= "mean")


    def forward(self, x):
        return self.model(x)

    def training_step(self,batch, batch_idx):
        x, y = batch
        logits = self(x) #logits already returned after log_softmax so as log-space distribution
        loss = self.criterion(logits, y)
        self.train_kl.update(y, torch.exp(logits)) #check if correct for log
        self.log_dict(
            {"train_loss": loss,
             "train_kl": self.train_kl},
            on_step= False, on_epoch= True, prog_bar= True)
        return loss

    def validation_step(self,batch, batch_idx):

        x, y = batch
        logits = self(x) #logits already returned after log_softmax so as log-space distribution
        loss = self.criterion(logits, y)
        self.val_kl.update(y, torch.exp(logits)) #check if correct for log
        self.log_dict(
            {"val_loss": loss,
             "val_kl": self.val_kl},
            on_step= False, on_epoch= True, prog_bar= True)
        return loss

    def configure_optimizers(self):
        #add schedulers?
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr = self.hparams.lr

        )
        return {
            "optimizer": optimizer
        }
